In [1]:
# 학번과 이름을 출력하세요
# 예시
# 학번 = '30261240'
# 이름 = '이현정'

학번 = '20251799'
이름 = '최하은'

print(학번, 이름)

20251799 최하은


In [6]:
#예제1
def deco(func):
    def inner():
        print('running inner()')
    return inner

@deco
def target():
    print("running target()")

target()


target


running inner()


<function __main__.deco.<locals>.inner()>

In [10]:
#예제1-1
def logger(func):
    def wrapper(*args, **kwargs):
        print(f"[LOG] {func.__name__} called with args={args}, kwargs={kwargs}")
        result = func(*args, **kwargs)
        print(f"[LOG] {func.__name__} returned {result}")
        return result
    return wrapper

@logger
def add(a,b):
    return a+b

In [14]:
#예제2
registry = []

def register(func):
    print('runnong register(%s)' % func)
    registry.append(func)
    return func

@register
def f1():
    print('running f1()')
@register
def f2():
    print('running f2()')

def f3():
    print('running f3()')

def main():
    print('running main()')
    print('registry ->', registry)
    f1()
    f2()
    f3()

if __name__=='__name__':
    main()

runnong register(<function f1 at 0x0000014EF0AC4B40>)
runnong register(<function f2 at 0x0000014EF0AC4D50>)


In [16]:
#예제3
promos = []
def promotion(promo_func):
    promos.append(promo_func)
    return promo_func

@promotion
def fidelity_promo(order):
    return order.total() * 0.05 if order.customer.fidelity >= 1000 else 0

@promotion
def bulk_item_promo(order):
    discount = 0
    for item in order.cart:
        if item.quantity >= 20:
            discount += item.total() * 0.1
    return discount

@promotion
def large_order_promo(order):
    distinct_items = {item.product for item in order.cart}
    if len(distinct_items) >= 10:
        return order.total() * 0.07
    return 0

def best_promo(order):
    return(max(promo(order)) for promo in promos)

In [17]:
#예제4
def f1(a):
    print(a)
    print(b)

f1(3)

3


NameError: name 'b' is not defined

In [18]:
def f1(a):
    print(a)
    print(b)
b=6
f1(3)

3
6


In [19]:
#예제5
b = 6
def f2(a):
    print(a)
    print(b)
    b = 9

f2(3)


3


UnboundLocalError: cannot access local variable 'b' where it is not associated with a value

In [20]:
#예제6
b = 6
def f3(a):
    global b
    print(a)
    print(b)
    b = 9

f3(3)


3
6


In [21]:
b

9

In [22]:
f3(3)

3
9


In [23]:
b = 30
f3(3)

3
30


In [27]:
#예제7
class Averager():
    def __init__(self):
        self.series = []

    def __call__(self, new_value):
        self.series.append(new_value)
        total = sum(self.series)
        return total/len(self.series)

avg = Averager()
avg(10)
avg(11)
avg(12)


11.0

In [28]:
#예제8
def make_averager():
    series = []

    def averager(new_value):
        series.append(new_value)
        total = sum(series)
        return total/len(series)

    return averager

avg = make_averager()
avg(10)
avg(11)
avg(12)

11.0

In [31]:
#예제9
avg.__code__.co_varnames


('new_value', 'total')

In [32]:
avg.__code__.co_freevars

('series',)

In [33]:
avg.__closure__

(<cell at 0x0000014EEF467160: list object at 0x0000014EF0CF4380>,)

In [34]:
avg.__closure__[0].cell_contents

[10, 11, 12]

In [35]:
#예제10
def make_averager():
    count = 0
    total = 0

    def averager(new_value):
        count += 1
        total += new_value
        return total / count

    return averager

avg = make_averager()
avg(10)

UnboundLocalError: cannot access local variable 'count' where it is not associated with a value

In [36]:
#예제11
def make_averager():
    count = 0
    total = 0

    def averager(new_value):
        nonlocal count, total
        count += 1
        total += new_value
        return total / count

    return averager

avg = make_averager()
avg(10)

10.0

In [38]:
#예제12
import time

def clock(func):
    def clocked(*args):
        t0 = time.perf_counter()
        result = func(*args)
        elapsed = time.perf_counter() - t0
        name = func.__name__
        arg_str = ', '.join(repr(arg) for arg in args)
        print('[%0.8fs] %s(%s) -> %r' % (elapsed, name, arg_str, result))
        return result
    return clocked


In [ ]:
#예제13
@clock
def snooze(seconds):
    time.sleep(seconds)

@clock
def factorial(n):
    return 1 if n < 2 else n*factorial(n-1)

if __name__=='__main__':
    print('*'*40, 'Calling snooze(.123)')
    snooze(.123)
    print('*'*40, 'Calling factorial(6)')
    print('6! =', factorial)

**************************************** Calling snooze(.123)
[0.12340510s] snooze(0.123) -> None
**************************************** Calling factorial(6)
[0.00000060s] factorial(1) -> 1
[0.00002010s] factorial(2) -> 2
[0.00003260s] factorial(3) -> 6
[0.00004410s] factorial(4) -> 24
[0.00005620s] factorial(5) -> 120
[0.00006900s] factorial(6) -> 720
6! = 720


In [43]:
#예제14
import time
import functools

def clocks(func):
    @functools.wraps(func)
    def clocked(*args, **kwargs):
        t0 = time.time()
        result = func(*args, **kwargs)
        elapsed = time.time() - t0
        arg_list = []
        name = func.__name__
        if args:
            arg_list.append(', '.join(repr(arg) for arg in args))
        if kwargs:
            pairs = ['%s=%r' % (k,w) for k,w in sorted(kwargs.items())]
            arg_list.append(', '.join(pairs))
        arg_str = ', '.join(arg_lst)
        print('[%0.8fs] %s(%s) -> %r' % (elapsed, name, arg_str, result))
        return result
    return clocked


@clock
def snooze(seconds):
    time.sleep(seconds)

@clock
def factorial(n):
    return 1 if n < 2 else n*factorial(n-1)

if __name__=='__main__':
    print('*'*40, 'Calling snooze(.123)')
    snooze(.123)
    print('*'*40, 'Calling factorial(6)')
    print('6! =', factorial)

**************************************** Calling snooze(.123)
[0.12336590s] snooze(0.123) -> None
**************************************** Calling factorial(6)
6! = <function clock.<locals>.clocked at 0x0000014EF0CAF690>
